# MediFlow Web Skin — 공개 데이터 후보 v1 패키징

Hair와 같이 선택 모델과 결과를 별도 Drive 폴더와 ZIP으로 보관합니다.
**재학습·중복 검사·Test 재평가를 하지 않습니다. GPU도 필요 없습니다.**

- 후보: EfficientNet-B0 / 256 / CE, Stage 2 최고 8번째 Epoch
- Validation 0.796, Test 0.8825, Test Macro F1 0.8813822927981046
- 정상 포함: 건선 → 아토피 → 여드름 → 정상 → 주사
- 기존 실험과 모델은 보존합니다. 실제 웹캠 검증 전 공개 데이터 후보입니다.

Drive에 완료된 suite 폴더가 있으면 **모두 실행**만 하면 됩니다.
전체 학습결과 ZIP을 다운로드하거나 모델을 따로 업로드할 필요 없습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip -q install tensorflow==2.20.0 keras==3.13.2
import keras
import tensorflow as tf
from pathlib import Path
if keras.__version__ != '3.13.2' or tf.__version__ != '2.20.0':
    raise RuntimeError('설치 버전 반영을 위해 런타임을 재시작한 뒤 모두 실행하세요.')
print('패키징 환경:', tf.__version__, keras.__version__, '(GPU 불필요)')


## 1. 검증·패키징 코드
실행 코드를 내장했습니다. 별도 저장소나 .py 업로드가 필요 없습니다.


In [ ]:
PACKAGING_SOURCE = '"""Preserve and package the already selected Web Skin candidate, without training."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport platform\nimport shutil\nimport uuid\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport keras\nimport numpy as np\nimport tensorflow as tf\n\nCLASSES = ["건선", "아토피", "여드름", "정상", "주사"]\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef write_json(path, value):\n    with Path(path).open("x", encoding="utf-8") as stream:\n        json.dump(value, stream, ensure_ascii=False, indent=2)\n\n\ndef within(root, relative):\n    path = (Path(root) / relative).resolve()\n    if not path.is_relative_to(Path(root).resolve()):\n        raise ValueError("Package path escapes its root")\n    return path\n\n\ndef verify_source(source, policy):\n    """Only accept the report bytes reviewed before packaging was requested."""\n    source = Path(source)\n    for relative, expected in policy["report_hashes"].items():\n        if sha256_file(within(source, relative)) != expected:\n            raise ValueError("Reviewed report changed: " + relative)\n    selection = read_json(source / "selection_before_test.json")\n    config = read_json(source / "suite_config.json")\n    card = read_json(source / "model_card.json")\n    records = read_json(source / "all_validation_results.json")\n    winner = max(records, key=lambda r: r["validation"]["accuracy"])\n    if (\n        selection["winner"] != "b0_256_ce"\n        or winner["id"] != selection["winner"]\n        or winner["validation"] != selection["validation"]\n        or card["model_sha256"] != policy["model_sha256"]\n        or selection["model_sha256"] != policy["model_sha256"]\n        or card["model_relative_path"] != policy["model_relative_path"]\n        or config["settings"]["class_names"] != CLASSES\n        or card["class_names"] != CLASSES\n        or config["settings"]["data_sha256"] != policy["data_sha256"]\n        or card["test"] != read_json(source / "final_test_metrics.json")\n    ):\n        raise ValueError("Source candidate contract mismatch")\n    model_path = within(source, policy["model_relative_path"])\n    if sha256_file(model_path) != policy["model_sha256"]:\n        raise ValueError("Selected model hash differs from the reviewed candidate")\n    return model_path, card\n\n\ndef check_model_contract(path):\n    model = keras.models.load_model(path, compile=False)\n    if (\n        tuple(model.input_shape) != (None, 256, 256, 3)\n        or tuple(model.output_shape) != (None, 5)\n        or model.count_params() != 4055976\n    ):\n        raise ValueError("Model shape or parameter count mismatch")\n    bases = [layer for layer in model.layers if isinstance(layer, keras.Model)]\n    if len(bases) != 1 or "efficientnetb0" not in bases[0].name.lower():\n        raise ValueError("Expected EfficientNet-B0")\n    rescaling = [layer for layer in bases[0].layers if isinstance(layer, keras.layers.Rescaling)]\n    if not any(\n        np.asarray(layer.scale).size == 1\n        and np.isclose(float(np.asarray(layer.scale).item()), 1 / 255)\n        and np.allclose(layer.offset, 0)\n        for layer in rescaling\n    ):\n        raise ValueError("Internal Rescaling(1/255) missing")\n    values = np.broadcast_to(\n        np.array([0, 127.5, 255], dtype="float32")[:, None, None, None], (3, 256, 256, 3)\n    ).copy()\n    scores = np.asarray(model(values, training=False))\n    if (\n        scores.shape != (3, 5)\n        or not np.isfinite(scores).all()\n        or np.any(scores < 0)\n        or np.any(scores > 1)\n        or not np.allclose(scores.sum(axis=1), 1, atol=1e-5)\n    ):\n        raise ValueError("Model output check failed")\n    return {\n        "input_shape": [256, 256, 3],\n        "output_count": 5,\n        "parameters": model.count_params(),\n        "dummy_forward_passed": True,\n        "note": "Artificial inputs test execution only; no new performance evaluation",\n    }\n\n\ndef build_package(source, output_parent, policy, packaging_source=None):\n    """Build locally; source files are immutable and no test images are loaded."""\n    source = Path(source)\n    model_path, card = verify_source(source, policy)\n    contract = check_model_contract(model_path)\n    name = (\n        "public_candidate_v1_b0_256_ce_"\n        + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")\n        + "_"\n        + uuid.uuid4().hex[:8]\n    )\n    package = Path(output_parent) / name\n    package.mkdir(parents=True, exist_ok=False)\n    shutil.copyfile(model_path, package / "web_skin_model.keras")\n    if sha256_file(package / "web_skin_model.keras") != policy["model_sha256"]:\n        raise ValueError("Copied model hash mismatch")\n    reports = package / "source_reports"\n    reports.mkdir()\n    write_json(package / "packaging_policy.json", policy)\n    if packaging_source is not None:\n        (package / "packaging_source.py").write_text(packaging_source, encoding="utf-8")\n    for relative, expected in policy["report_hashes"].items():\n        target = within(reports, relative)\n        target.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copyfile(within(source, relative), target)\n        if sha256_file(target) != expected:\n            raise ValueError("Copied report hash mismatch: " + relative)\n    write_json(package / "class_names.json", CLASSES)\n    write_json(\n        package / "preprocessing.json",\n        {\n            "input_shape": [256, 256, 3],\n            "color_order": "RGB",\n            "input_dtype": "float32",\n            "input_pixel_range": [0, 255],\n            "external_normalization": False,\n            "internal_rescaling": "1/255",\n            "resize": "TensorFlow bilinear, antialias=False",\n            "aspect_ratio": "resize to 256x256, no crop/pad",\n            "exif_transpose": False,\n            "output": "5 softmax scores in class_names.json order; not calibrated correctness",\n        },\n    )\n    write_json(package / "model_contract_check.json", contract)\n    manifest = {\n        "status": "public_data_candidate_v1_not_device_validated",\n        "domain": "web_skin",\n        "created_at": datetime.now(timezone.utc).isoformat(),\n        "source_suite": source.name,\n        "model_file": "web_skin_model.keras",\n        "model_sha256": policy["model_sha256"],\n        "model_size_bytes": (package / "web_skin_model.keras").stat().st_size,\n        "model_parameter_count": contract["parameters"],\n        "backbone": "EfficientNet-B0",\n        "image_size": [256, 256],\n        "loss": "Categorical Crossentropy",\n        "stage1_epochs": 15,\n        "stage2_epochs_run": 10,\n        "selected_stage2_epoch": 8,\n        "selection_metric": "validation_accuracy; ties keep earlier experiment",\n        "validation_accuracy": card["validation"]["accuracy"],\n        "test_accuracy": card["test"]["accuracy"],\n        "macro_f1": card["test"]["macro_f1"],\n        "data_zip_sha256": policy["data_sha256"],\n        "normal_class_included": True,\n        "class_names_file": "class_names.json",\n        "preprocessing_file": "preprocessing.json",\n        "validation": card["validation"],\n        "test": card["test"],\n        "limitations": card["limitations"],\n        "new_training_or_test_evaluation": False,\n        "packaging_environment": {\n            "python": platform.python_version(),\n            "keras": keras.__version__,\n            "tensorflow": tf.__version__,\n            "numpy": np.__version__,\n        },\n    }\n    (package / "MODEL_CARD.md").write_text(\n        "# MediFlow Web Skin 공개 데이터 후보 v1\\n\\n"\n        "EfficientNet-B0 / 256 / CE. 실제 웹캠 검증 전 공개 데이터 후보입니다.\\n\\n"\n        f"클래스 순서: {CLASSES}\\n\\n"\n        f"Validation Accuracy: {manifest[\'validation_accuracy\']}\\n\\n"\n        f"Test Accuracy: {manifest[\'test_accuracy\']}\\n\\n"\n        f"Test Macro F1: {manifest[\'macro_f1\']}\\n\\n"\n        "위 수치는 완료된 실험의 보고값을 그대로 보존한 것으로 새 평가가 아닙니다.\\n"\n        "정상 클래스는 포함하지만 범위 밖 입력 거부 기능은 없습니다.\\n"\n        "사람·병변·촬영 세션 누수 및 실제 웹캠 성능은 미검증입니다.\\n"\n        "입력은 얼굴 정면 RGB float32 0~255이며 외부 /255를 적용하지 않습니다.\\n"\n        "모델 출력은 보정되지 않은 점수입니다. 자세한 계약은 preprocessing.json을 보세요.\\n",\n        encoding="utf-8",\n    )\n    manifest["artifact_hashes"] = {\n        p.relative_to(package).as_posix(): sha256_file(p)\n        for p in sorted(package.rglob("*"))\n        if p.is_file()\n    }\n    write_json(package / "package_manifest.json", manifest)\n    archive_path = package.with_suffix(".zip")\n    with zipfile.ZipFile(archive_path, "x", compression=zipfile.ZIP_DEFLATED) as archive:\n        for p in sorted(package.rglob("*")):\n            if p.is_file():\n                archive.write(p, arcname=name + "/" + p.relative_to(package).as_posix())\n    with zipfile.ZipFile(archive_path) as archive:\n        if archive.testzip() is not None:\n            raise ValueError("Package ZIP CRC failure")\n        expected = {\n            **manifest["artifact_hashes"],\n            "package_manifest.json": sha256_file(package / "package_manifest.json"),\n        }\n        for relative, digest in expected.items():\n            with archive.open(name + "/" + relative) as stream:\n                actual = hashlib.sha256()\n                for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n                    actual.update(chunk)\n                if actual.hexdigest() != digest:\n                    raise ValueError("ZIP content hash mismatch: " + relative)\n    return package, archive_path\n\n\ndef publish_package(package, archive_path, drive_parent):\n    """Copy only verified outputs, preserving every existing Drive package."""\n    package, archive_path, drive_parent = map(Path, (package, archive_path, drive_parent))\n    drive_parent.mkdir(parents=True, exist_ok=True)\n    target = drive_parent / package.name\n    destination = drive_parent / archive_path.name\n    if target.exists() or destination.exists():\n        raise FileExistsError("Destination already exists; nothing overwritten")\n    shutil.copytree(package, target)\n    for path in package.rglob("*"):\n        if path.is_file() and sha256_file(path) != sha256_file(target / path.relative_to(package)):\n            raise ValueError("Drive package copy verification failed")\n    with archive_path.open("rb") as src, destination.open("xb") as dst:\n        shutil.copyfileobj(src, dst)\n    digest = sha256_file(archive_path)\n    if sha256_file(destination) != digest:\n        raise ValueError("Drive ZIP copy verification failed")\n    with destination.with_suffix(".zip.sha256").open("x", encoding="utf-8") as stream:\n        stream.write(digest + "  " + destination.name + "\\n")\n    return target, destination, digest\n'
exec(compile(PACKAGING_SOURCE, "candidate_packaging.py", "exec"))


## 2. 완료된 결과 위치
결과 폴더를 이동하지 않았다면 그대로 실행하세요. 검토한 결과/모델과 다르면 중단합니다.


In [ ]:
SOURCE_SUITE = Path('/content/drive/MyDrive/mediflow_experiments/web_skin/suite_20260908_014452_72768a42')
DRIVE_OUTPUT = Path('/content/drive/MyDrive/mediflow_models/web_skin')
LOCAL_OUTPUT = Path('/content/mediflow_web_skin_packages')
POLICY = {'model_relative_path': 'b0_256_ce/attempt_8977c45d04bc/stage2_best.keras', 'model_sha256': 'd4c834a7fd47480c1c45ddbdc09257942b38b2408d5b28effc34ee68c4fad97b', 'data_sha256': 'f8908af3d54e521ad14c37a44b569d33fe92be3b8b9b66a8d80faf4ba964072d', 'report_hashes': {'all_training_curves.png': '70c14bef7c1029e99b0f9c9e38562891afa15b58b764bcc492da7b677eeab96e', 'all_validation_confusion_matrices.png': 'c310000005c67c1bc9a42477982ed5b5d1544fd5430cb36e99d92da4c316a4f7', 'all_validation_results.json': '6927b1336d11f55256e223ff7b5cc77827f4568d95fe8b2d04a4c2a5b5aaed0a', 'b0_256_ce/attempt_8977c45d04bc/record.json': 'a85b9059745c64143d17ee3fc4fcd14caa8b3bb8e82bf68e0516fa767f3c8290', 'b0_256_ce/attempt_8977c45d04bc/spec.json': '9469bcff367946b0863ef1aa3658231b9e5904bab98f790b1c31f95a3b03cb98', 'b0_256_ce/attempt_8977c45d04bc/stage1_history.json': '893f4a06854fcc2c8520cba6f81f8510d02c9fa009ae109710355cdda8e2d2e3', 'b0_256_ce/attempt_8977c45d04bc/stage1_log.csv': '60448a11cca11e9bc6be3e603b9063650b55ba9b52967cbf63eed2e354aecce6', 'b0_256_ce/attempt_8977c45d04bc/stage2_history.json': 'd356029838f7af2c9fb8d317397aa67101263551bf4ca999b4d9bc3b9a495984', 'b0_256_ce/attempt_8977c45d04bc/stage2_log.csv': '3b962c08b9e977a5a32d1633567ffdb4c38ab004a5d3c5a22e3c6acc9afa3e35', 'b0_256_ce/attempt_8977c45d04bc/training_curves.png': 'c09741c1e83e7a7a46ce5bb5dc4b9dd0d0d98a3ad490347ffcd392f09d23e58a', 'b0_256_ce/attempt_8977c45d04bc/validation_metrics.json': '0db115144157dd134a335f9900222b5eb95528f6c6df8066b12362edc7a40c6e', 'b0_256_ce/attempt_8977c45d04bc/validation_predictions.csv': '1d2e32bcd1d6d274ff2bc65c74f9cf08b810c549509f99f008b8a4691b0ed8d5', 'b0_256_ce/completed.json': '9f52b54e0b8cc3a80c1cadf22c873ae81fdaf2e7b464536a0b8e0ca692775dfb', 'class_mapping.json': 'e0dee8e1deac1f5ccda2729c6d107b4499d901a2dac9542649e963d714b09658', 'data_source.json': '584220774aa3b98c9505a7b11638cbe7fb0fe58564e919ba349e3a4e79b0cea6', 'environment_20260908_014452_72768a42.json': '399fa6f309930cc4b8ed63482bd2bc1880a099bb5c135ad5abb91b9c2593486b', 'experiment_comparison.csv': 'e7a25033abd687151bc26fd3f7cfd4d0a1b3f4c57299b6eb4b7e76e9c4accfe4', 'experiment_suite.py': 'bb9bf2d355ce68753275c63e30c2a1958e4df1ef7a1df3797ab33d32edf657a1', 'final_test_confusion_matrix.png': 'd5ef9f3ca94fdfc83a98cbe9be9e3f441e31bbdcc17ccfefb393584af00675a5', 'final_test_metrics.json': 'af9e77f41608eeb73acb249079b694f11184f9095d523be5daa9ebdbabe3334c', 'final_test_predictions.csv': '452a734db2091ff772b1bf1708e14610a12c8f59b02f96c769536db7af179e13', 'model_card.json': '153000d1173d36c0e7ce4aedc26d3b8a40eccc2b2fec33ecbcfaf755885b77d7', 'notebook_snapshot_20260908_014452_72768a42.py': '5052600e4e438ab00395d7701ab5ddda99f505cbbf15867d45bbef54a87945f8', 'selection_before_test.json': '884f6cf4b6b66561828fce52ae41deef2d574ac41eafd611bfc7696c71291b13', 'suite_config.json': '441134c323fe388e8496c5ef8288b963c570e48908efadff6955fd1fec196f72', 'test_completed.json': '9b8939d0667fd34e8ac581240d1888ac5502e093fac0289c4d605f6cf5db46a7', 'validation_performance_dashboard.png': '0478939c3d2c4c4f336f77f5fe73a8221625052d72a57b29ad72fdda5a6f6bfa'}}
print('원본 실험:', SOURCE_SUITE)
print('후보 저장 위치:', DRIVE_OUTPUT)


## 3. 후보 검증 및 패키지 생성
검토 당시 보고서의 식별값, 모델 SHA-256, 클래스 순서, 입출력, 내부 정규화를 확인합니다.
가상 입력의 출력 확인은 실행 점검이며 성능 재평가가 아닙니다.


In [ ]:
local_package, local_zip = build_package(
    SOURCE_SUITE, LOCAL_OUTPUT, POLICY, packaging_source=PACKAGING_SOURCE)
print('검증과 로컬 ZIP 생성 완료:', local_zip)


## 4. Drive 저장 및 복사 확인
후보 모델 한 개와 비교 그래프·평가 JSON/CSV·모델 카드·입출력 정보·무결성 목록이 저장됩니다.


In [ ]:
package_path, zip_path, zip_sha256 = publish_package(local_package, local_zip, DRIVE_OUTPUT)
print('Drive 패키지:', package_path)
print('보내줄 ZIP:', zip_path)
print('ZIP bytes:', zip_path.stat().st_size)
print('ZIP SHA-256:', zip_sha256)
print('모델 SHA-256:', POLICY['model_sha256'])
print('완료: 위 후보 ZIP을 다운로드해서 보내주세요.')


## 완료 후
출력된 `public_candidate_v1_b0_256_ce_...zip`을 보관하고 공유하세요.
모든 Epoch 모델이 아닌 **선정 모델 1개**가 들어갑니다. 해시 파일 `.zip.sha256`도 함께 보관합니다.

폴더/ZIP 복사 오류가 나면 완료로 간주하지 않습니다. 원본 실험은 그대로 남으며 재실행은 새 이름으로 저장합니다.
실제 웹캠 평가와 시스템 모델 교체는 이 패키징에 포함되지 않습니다.
